# PMM Dynamic Screener — NonKYC Public REST

This notebook screens **NONKYC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

Default quote asset is `USDT`, but you can switch `QUOTE_ASSET` to `XMR`, `BTC`, or any other quote supported by NonKYC. The notebook uses the documented public REST endpoints for markets, tickers, order books, candles, and trades.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [ ]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import NonKYCPublicScreener, default_nonkyc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.nonkyc_public import NONKYC_BASE_URL


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [ ]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "USDT"
INTERVAL = "5m"
UNIVERSE_TOP_K = 80
FINAL_TOP_N = 25
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "nonkyc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_nonkyc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 100000.0
cfg.max_spread_bps = 90.0
cfg.min_top_of_book_quote = 75.0
cfg.min_depth_10bps_quote = 250.0
cfg.max_last_trade_age_sec = 3600.0
cfg.min_recent_trade_count = 40
cfg.min_candle_count = 220
cfg.min_candle_coverage_ratio = 0.95
cfg.max_zero_volume_fraction = 0.3
cfg.min_natr_bps = 12.0
cfg.max_natr_bps = 400.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [ ]:
screener = NonKYCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [ ]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [ ]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [ ]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=NONKYC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [ ]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())
